# Deep Retention — Análisis Exploratorio de Datos

EDA académico del archivo `data/Customer-Churn-Records.csv`, centrado en diagnosticar la calidad y comprender las variables antes del futuro entrenamiento de una red neuronal MLP.

Este notebook analiza directamente el DataFrame cargado y no aplica limpieza ni transformaciones. No modifica el CSV, no implementa el preprocesamiento y no entrena modelos.

## 1. Carga e inspección del dataset

El archivo se localiza con `pathlib` mediante rutas relativas. La carga funciona al ejecutar Jupyter desde la raíz del proyecto o desde `notebooks/`, sin guardar rutas absolutas.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

DATA_FILENAME = "Customer-Churn-Records.csv"
DATA_CANDIDATES = [
    Path("data") / DATA_FILENAME,
    Path("..") / "data" / DATA_FILENAME,
]
DATA_PATH = next((path for path in DATA_CANDIDATES if path.is_file()), None)
if DATA_PATH is None:
    raise FileNotFoundError(f"No se encontró {DATA_FILENAME} desde la raíz ni desde notebooks/.")

PREDICTORS = [
    "CreditScore", "Geography", "Age", "Tenure", "Balance",
    "NumOfProducts", "HasCrCard", "IsActiveMember", "EstimatedSalary",
]
TARGET = "Exited"
IDENTIFICATION_COLUMNS = ["CustomerId", "Surname", "Gender"]
FINANCIAL_COLUMNS = [
    "MonthlyTransactions_Sim", "MonthlyTransactionVolume_USD_Sim",
    "AvgTransactionFee_USD_Sim", "AnnualCommissionIncome_USD_Sim",
    "AnnualFinancialMargin_USD_Sim", "AnnualOperatingCosts_USD_Sim",
    "AnnualNetMargin_USD_Sim", "ProfitabilitySegment_Sim",
]

In [3]:

# Cargar el nuevo dataset
df = pd.read_csv(DATA_PATH)

# Mostrar las columnas existentes
print("Columnas del nuevo dataset:")
print(df.columns.tolist())

# Verificar únicamente las variables necesarias
required_columns = (
    PREDICTORS
    + [TARGET]
    + ["CustomerId"]
)

missing_columns = sorted(
    set(required_columns) - set(df.columns)
)

if missing_columns:
    raise ValueError(
        f"Faltan columnas requeridas: {missing_columns}"
    )

print(f"\nArchivo: {DATA_PATH.as_posix()}")
print(
    f"Dimensiones: {df.shape[0]:,} filas "
    f"x {df.shape[1]} columnas"
)

display(df.head())
df.info()


Columnas del nuevo dataset:
['RowNumber', 'CustomerId', 'Surname', 'CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited', 'Complain', 'Satisfaction Score', 'Card Type', 'Point Earned']

Archivo: ../data/Customer-Churn-Records.csv
Dimensiones: 10,000 filas x 18 columnas


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Complain,Satisfaction Score,Card Type,Point Earned
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1,1,2,DIAMOND,464
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,1,3,DIAMOND,456
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,1,3,DIAMOND,377
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0,0,5,GOLD,350
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0,0,5,GOLD,425


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   RowNumber           10000 non-null  int64  
 1   CustomerId          10000 non-null  int64  
 2   Surname             10000 non-null  object 
 3   CreditScore         10000 non-null  int64  
 4   Geography           10000 non-null  object 
 5   Gender              10000 non-null  object 
 6   Age                 10000 non-null  int64  
 7   Tenure              10000 non-null  int64  
 8   Balance             10000 non-null  float64
 9   NumOfProducts       10000 non-null  int64  
 10  HasCrCard           10000 non-null  int64  
 11  IsActiveMember      10000 non-null  int64  
 12  EstimatedSalary     10000 non-null  float64
 13  Exited              10000 non-null  int64  
 14  Complain            10000 non-null  int64  
 15  Satisfaction Score  10000 non-null  int64  
 16  Card 

### Variables del análisis predictivo

- **Predictoras:** `CreditScore`, `Geography`, `Age`, `Tenure`, `Balance`, `NumOfProducts`, `HasCrCard`, `IsActiveMember` y `EstimatedSalary`.
- **Objetivo:** `Exited`.
- **Identificación y presentación, excluidas del MLP:** `CustomerId`, `Surname` y `Gender`.
- **Financieras simuladas, excluidas del MLP:** las ocho columnas terminadas en `_Sim`.

Esta separación impide utilizar identificadores, datos de presentación o variables reservadas para el motor financiero como entradas de la red neuronal.

## 2. Diagnóstico de calidad de los datos

Se revisan valores faltantes, duplicados, unicidad del identificador, tipos, rangos y dominios binarios sin modificar los datos. Los problemas detectados se documentan para tratarlos posteriormente en `02_preprocesamiento.ipynb`.

In [4]:
missing_by_column = df.isna().sum().rename("valores_nulos").to_frame()
display(missing_by_column)

quality_summary = pd.DataFrame({
    "control": [
        "Valores faltantes", "Filas duplicadas", "CustomerId duplicados",
        "CustomerId únicos", "Apellidos con '?'",
    ],
    "resultado": [
        int(df.isna().sum().sum()),
        int(df.duplicated().sum()),
        int(df["CustomerId"].duplicated().sum()),
        int(df["CustomerId"].nunique()),
        int(df["Surname"].str.contains("?", regex=False, na=False).sum()),
    ],
})
display(quality_summary)

,valores_nulos
RowNumber,0
CustomerId,0
Surname,0
CreditScore,0
Geography,0
Gender,0
Age,0
Tenure,0
Balance,0
NumOfProducts,0


,control,resultado
0,Valores faltantes,0
1,Filas duplicadas,0
2,CustomerId duplicados,0
3,CustomerId únicos,10000
4,Apellidos con '?',92


In [5]:
expected_numeric_columns = (
    ["CustomerId"]
    + [column for column in PREDICTORS if column != "Geography"]
    + [TARGET]
    + FINANCIAL_COLUMNS[:-1]
)
numeric_type_check = pd.DataFrame({
    "tipo": df[expected_numeric_columns].dtypes.astype(str),
    "es_numérica": [pd.api.types.is_numeric_dtype(df[column]) for column in expected_numeric_columns],
})
display(numeric_type_check)

range_rules = {
    "CreditScore": (300, 850),
    "Age": (18, 120),
    "Tenure": (0, 10),
    "Balance": (0, None),
    "NumOfProducts": (1, 4),
    "EstimatedSalary": (0, None),
}
range_rows = []
for column, (lower, upper) in range_rules.items():
    invalid = df[column] < lower
    if upper is not None:
        invalid = invalid | (df[column] > upper)
    criterion = f"{lower} ≤ valor" if upper is None else f"{lower} ≤ valor ≤ {upper}"
    range_rows.append({
        "variable": column,
        "mínimo": df[column].min(),
        "máximo": df[column].max(),
        "criterio": criterion,
        "fuera_de_rango": int(invalid.sum()),
    })
display(pd.DataFrame(range_rows))

binary_rows = []
for column in ["HasCrCard", "IsActiveMember", TARGET]:
    binary_rows.append({
        "variable": column,
        "valores_observados": sorted(df[column].dropna().unique().tolist()),
        "valores_inválidos": int((~df[column].isin([0, 1]) & df[column].notna()).sum()),
    })
display(pd.DataFrame(binary_rows))

KeyError: "['MonthlyTransactions_Sim', 'MonthlyTransactionVolume_USD_Sim', 'AvgTransactionFee_USD_Sim', 'AnnualCommissionIncome_USD_Sim', 'AnnualFinancialMargin_USD_Sim', 'AnnualOperatingCosts_USD_Sim', 'AnnualNetMargin_USD_Sim'] not in index"

**Interpretación.** El dataset original no contiene valores faltantes ni filas duplicadas y sus 10.000 identificadores son únicos. Se detectan 92 apellidos que contienen `?`, lo que indica caracteres desconocidos. El EDA solo informa esta situación: los apellidos no se modifican y su tratamiento se realiza sobre una copia en `02_preprocesamiento.ipynb`. Los tipos numéricos, rangos revisados y dominios binarios son consistentes.

## 3. Distribución y desbalance de `Exited`

`Exited = 0` indica permanencia y `Exited = 1` abandono. La proporción de clases orientará la partición y las métricas del futuro modelo.

In [ ]:
target_distribution = df[TARGET].value_counts().sort_index().to_frame("cantidad")
target_distribution["porcentaje"] = target_distribution["cantidad"] / len(df) * 100
target_distribution.index = ["Permaneció (0)", "Abandonó (1)"]
display(target_distribution)

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(target_distribution.index, target_distribution["cantidad"], color=["#4C78A8", "#E45756"])
ax.bar_label(bars, labels=[f"{count:,} ({pct:.2f} %)" for count, pct in zip(
    target_distribution["cantidad"], target_distribution["porcentaje"]
)], padding=3)
ax.set(title="Distribución histórica de Exited", xlabel="Clase", ylabel="Clientes")
ax.set_ylim(0, target_distribution["cantidad"].max() * 1.12)
plt.tight_layout()
plt.show()

**Interpretación.** El 20,37 % de los clientes abandonó y el 79,63 % permaneció, una relación aproximada de 3,91 a 1. En la siguiente fase la separación deberá ser estratificada y se valorará la ponderación de clases; la exactitud por sí sola no será una métrica suficiente.

## 4. Estadísticas y distribuciones de las variables predictoras

Las estadísticas se limitan a las características del futuro MLP. `CustomerId`, `Surname`, `Gender`, `Exited` y las variables financieras simuladas quedan fuera.

In [ ]:
numeric_predictors = [column for column in PREDICTORS if column != "Geography"]
continuous_predictors = ["CreditScore", "Age", "Tenure", "Balance", "EstimatedSalary"]
display(df[continuous_predictors].describe().T)

frequency_tables = []
for column in ["Geography", "NumOfProducts", "HasCrCard", "IsActiveMember"]:
    table = df[column].value_counts().sort_index().rename("cantidad").reset_index()
    table.columns = ["categoria", "cantidad"]
    table.insert(0, "variable", column)
    frequency_tables.append(table)

display(pd.concat(frequency_tables, ignore_index=True))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, column in zip(axes.flat, continuous_predictors):
    ax.boxplot(df[column].dropna(), orientation="horizontal")
    ax.set(title=column, xlabel=column)
    ax.set_yticks([])
axes.flat[-1].axis("off")
fig.suptitle("Diagramas de caja para revisión de valores atípicos", fontsize=14)
plt.tight_layout()
plt.show()

**Interpretación.** Los diagramas de caja horizontales permiten revisar de forma sencilla la dispersión y los posibles valores atípicos. Se observan puntos extremos principalmente en `CreditScore` y `Age`; se conservan porque un valor alejado del resto no es necesariamente un dato incorrecto. Las distintas escalas se tratarán después de separar los conjuntos de datos.

### Clientes con saldo cero

Se comprueba por separado `Balance = 0` porque es un valor válido y frecuente, no un dato faltante.

In [ ]:
balance_zero_count = int(df["Balance"].eq(0).sum())
balance_summary = pd.DataFrame({
    "grupo": ["Saldo cero", "Saldo positivo"],
    "clientes": [balance_zero_count, len(df) - balance_zero_count],
})
balance_summary["porcentaje"] = balance_summary["clientes"] / len(df) * 100
display(balance_summary)

**Interpretación.** El 36,17 % de los clientes tiene `Balance = 0`. Como representa una condición posible de la cuenta y aparece en una proporción importante del dataset, se conserva sin convertirlo en valor faltante ni eliminar esos registros.

## 5. Matriz de correlación

Se calcula la correlación de Pearson entre las ocho predictoras numéricas y `Exited`. Se excluyen `CustomerId`, `Surname`, `Gender`, `Geography` y las variables financieras simuladas. `Geography` es nominal y su codificación corresponde a la fase posterior de preprocesamiento.

In [ ]:
correlation_columns = numeric_predictors + [TARGET]
correlation_matrix = df[correlation_columns].corr(method="pearson")
fig, ax = plt.subplots(figsize=(10, 8))
image = ax.imshow(correlation_matrix, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(correlation_columns)), correlation_columns, rotation=45, ha="right")
ax.set_yticks(range(len(correlation_columns)), correlation_columns)

for row in range(len(correlation_columns)):
    for column in range(len(correlation_columns)):
        value = correlation_matrix.iloc[row, column]
        text_color = "white" if abs(value) >= 0.55 else "black"
        ax.text(column, row, f"{value:.2f}", ha="center", va="center", color=text_color, fontsize=8)

colorbar = fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
colorbar.set_label("Correlación de Pearson")
ax.set_title("Matriz de correlación de predictoras numéricas y Exited")
plt.tight_layout()
plt.show()

**Interpretación.** Una correlación positiva indica que ambas variables tienden a aumentar juntas; una negativa indica que una tiende a disminuir cuando la otra aumenta. Respecto a `Exited`, la mayor asociación lineal positiva corresponde a `Age` (`r = 0,285`), seguida por `Balance` (`r = 0,119`), mientras que `IsActiveMember` presenta la asociación negativa de mayor magnitud (`r = -0,156`).

Las correlaciones restantes son débiles. Esto no implica que las variables sean inútiles para el MLP: la correlación de Pearson solo mide relaciones lineales entre pares y no captura relaciones no lineales ni interacciones entre características. Por ello, no se elimina ninguna predictora usando únicamente esta matriz.

## 6. Conclusiones y recomendaciones

### Conclusiones

- El dataset contiene 10.000 registros, sin valores faltantes ni filas duplicadas; los 10.000 `CustomerId` son únicos.
- Se detectaron 92 apellidos con `?`. El EDA no los modifica: la limpieza se delega a `02_preprocesamiento.ipynb`.
- Las columnas numéricas tienen tipos adecuados; los rangos revisados y las variables binarias son consistentes.
- `Exited` presenta un desbalance moderado: 20,37 % de abandono frente a 79,63 % de permanencia.
- El 36,17 % de los clientes tiene saldo cero, una condición válida que debe conservarse.
- Los boxplots horizontales muestran posibles valores atípicos, especialmente en `CreditScore` y `Age`; no se eliminan automáticamente porque requieren revisión contextual.
- En la matriz, `Age` tiene la mayor correlación positiva con `Exited`, `IsActiveMember` la mayor correlación negativa y `Balance` una asociación positiva menor. La correlación no implica causalidad ni determina por sí sola la utilidad de una variable.

### Recomendaciones para el preprocesamiento

1. Aplicar la limpieza sobre una copia: retirar espacios exteriores y representar los apellidos con `?` mediante `pd.NA`, sin eliminar clientes.
2. Separar entrenamiento, validación y prueba de forma estratificada antes de ajustar transformadores.
3. Ajustar el escalador y el codificador exclusivamente con entrenamiento para evitar *data leakage*.
4. Codificar `Geography` sin imponer un orden artificial y conservar las variables binarias validadas.
5. No eliminar automáticamente saldos cero ni observaciones señaladas visualmente por los boxplots.
6. Excluir de la matriz predictora `CustomerId`, `Surname`, `Gender`, `Exited` y las variables financieras simuladas.

Este notebook finaliza en el diagnóstico exploratorio. No crea un DataFrame depurado, no transforma variables y no entrena modelos.